<a href="https://colab.research.google.com/github/asthaa717/analisis-multivariat/blob/main/praktik_analisis_multivariat_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.datasets import load_linnerud
import pandas as pd
import numpy as np
data = load_linnerud()
X = pd.DataFrame(data.data,
                 columns=data.feature_names)
Y = pd.DataFrame(data.target,
                 columns=data.target_names)
print(X.shape)
print(Y.shape)

(20, 3)
(20, 3)


In [3]:
print(X.isna().sum())
print(Y.isna().sum())
print(X.describe())
print(Y.describe())

Chins     0
Situps    0
Jumps     0
dtype: int64
Weight    0
Waist     0
Pulse     0
dtype: int64
           Chins      Situps      Jumps
count  20.000000   20.000000   20.00000
mean    9.450000  145.550000   70.30000
std     5.286278   62.566575   51.27747
min     1.000000   50.000000   25.00000
25%     4.750000  101.000000   39.50000
50%    11.500000  122.500000   54.00000
75%    13.250000  210.000000   85.25000
max    17.000000  251.000000  250.00000
           Weight      Waist      Pulse
count   20.000000  20.000000  20.000000
mean   178.600000  35.400000  56.100000
std     24.690505   3.201973   7.210373
min    138.000000  31.000000  46.000000
25%    160.750000  33.000000  51.500000
50%    176.000000  35.000000  55.000000
75%    191.500000  37.000000  60.500000
max    247.000000  46.000000  74.000000


In [4]:
within_x = X.corr()
within_y = Y.corr()
combined = pd.concat([X, Y], axis=1)
cross = combined.corr().loc[
    X.columns, Y.columns]
print(within_x.round(3))
print(within_y.round(3))
print(cross.round(3))

        Chins  Situps  Jumps
Chins   1.000   0.696  0.496
Situps  0.696   1.000  0.669
Jumps   0.496   0.669  1.000
        Weight  Waist  Pulse
Weight   1.000  0.870 -0.366
Waist    0.870  1.000 -0.353
Pulse   -0.366 -0.353  1.000
        Weight  Waist  Pulse
Chins   -0.390 -0.552  0.151
Situps  -0.493 -0.646  0.225
Jumps   -0.226 -0.191  0.035


In [5]:
from sklearn.preprocessing import StandardScaler

sx = StandardScaler()
sy = StandardScaler()

X_scaled = sx.fit_transform(X)
Y_scaled = sy.fit_transform(Y)

In [6]:
from sklearn.cross_decomposition import CCA
cca = CCA(n_components=3,
          scale=False, max_iter=1000)
U, V = cca.fit_transform(
    X_scaled, Y_scaled)
r = np.array([
    np.corrcoef(U[:,i], V[:,i])[0,1]
    for i in range(3)])
print(np.round(r, 3))

[0.796 0.201 0.073]


In [7]:
x_loadings = pd.DataFrame(
    {f"CV{{i+1}}": [
        np.corrcoef(X_scaled[:,j],
                    U[:,i])[0,1]
        for j in range(3)]
     for i in range(3)},
    index=X.columns)
y_loadings = pd.DataFrame(
    {f"CV{{i+1}}": [
        np.corrcoef(Y_scaled[:,j],
                    V[:,i])[0,1]
        for j in range(3)]
     for i in range(3)},
    index=Y.columns)
print(x_loadings.round(3))
print(y_loadings.round(3))

        CV{i+1}
Chins     0.644
Situps   -0.054
Jumps     0.234
        CV{i+1}
Weight    0.135
Waist     0.031
Pulse    -0.942


In [8]:
x_cross = pd.DataFrame(
    {f"CV{{i+1}}": [
        np.corrcoef(X_scaled[:,j],
                    V[:,i])[0,1]
        for j in range(3)]
     for i in range(3)},
    index=X.columns)
y_cross = pd.DataFrame(
    {f"CV{{i+1}}": [
        np.corrcoef(Y_scaled[:,j],
                    U[:,i])[0,1]
        for j in range(3)]
     for i in range(3)},
    index=Y.columns)
print(x_cross.round(3))
print(y_cross.round(3))

        CV{i+1}
Chins     0.047
Situps   -0.004
Jumps     0.017
        CV{i+1}
Weight    0.010
Waist     0.002
Pulse    -0.068


In [10]:
# Re-calculate correct loadings with proper column names
x_loadings = pd.DataFrame(
    {f"CV{i+1}": [
        np.corrcoef(X_scaled[:,j],
                    U[:,i])[0,1]
        for j in range(3)]
     for i in range(3)},
    index=X.columns)

y_loadings = pd.DataFrame(
    {f"CV{i+1}": [
        np.corrcoef(Y_scaled[:,j],
                    V[:,i])[0,1]
        for j in range(3)]
     for i in range(3)},
    index=Y.columns)

# Calculate Variance Extracted (VE)
ve_x = (x_loadings**2).mean(axis=0)
ve_y = (y_loadings**2).mean(axis=0)

# Calculate Redundancy
red_x = ve_x * r**2
red_y = ve_y * r**2

# Create summary DataFrame
summary = pd.DataFrame({
    "Canonical_r": r,
    "r_squared": r**2,
    "VE_X": ve_x,
    "VE_Y": ve_y,
    "Redundancy_X": red_x,
    "Redundancy_Y": red_y
})
print(summary.round(3))

     Canonical_r  r_squared   VE_X   VE_Y  Redundancy_X  Redundancy_Y
CV1        0.796      0.633  0.408  0.451         0.258         0.285
CV2        0.201      0.040  0.434  0.247         0.017         0.010
CV3        0.073      0.005  0.157  0.302         0.001         0.002


In [12]:
rng = np.random.default_rng(42)
observed_r = r[0]
n_perm = 1000
perm_r = []
for _ in range(n_perm):
    idx = rng.permutation(len(Y_scaled))
    Y_perm = Y_scaled[idx]
    cca_perm = CCA(n_components=1,
                   scale=False, max_iter=1000)
    U_perm, V_perm = cca_perm.fit_transform(X_scaled, Y_perm)
    rp = np.corrcoef(U_perm[:,0], V_perm[:,0])[0,1]
    perm_r.append(rp)
perm_r = np.array(perm_r)
p_perm = (np.sum(perm_r >= observed_r) + 1) / (n_perm + 1)
print("Observed:", observed_r)
print("Permutation p-value:", p_perm)

Observed: 0.7956081520064181
Permutation p-value: 0.04095904095904096
